# Version 2 Model Run and Review

Run one model at a time, inspect the saved outputs quickly, and view the recommended stocks for the next period. This notebook is written to work in Google Colab when `version_2` is located at `/content/version_2`.


## Setup Root

Purpose: point the notebook to the model project root.


In [ ]:
from pathlib import Path
import subprocess, sys
import pandas as pd
from IPython.display import display

ROOT = Path('/content/version_2/model')
if not ROOT.exists():
    ROOT = Path.cwd().resolve().parents[0]
CONFIG = ROOT / 'configs' / 'default.yaml'
ROOT


## Install Requirements and Load Helpers

Purpose: install dependencies and load the small helper functions used by the run/review/recommend blocks.


In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(ROOT / 'requirements.txt')], check=True)
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))

from v2_model.config import load_config
from v2_model.recommend import build_latest_recommendations
from v2_model.recommend_true_latest import build_true_latest_recommendations

LAST_RUN_DIRS = {}

def _parse_run_dir(stdout_text: str):
    marker = 'Pipeline completed. Outputs saved to:'
    for line in reversed(stdout_text.splitlines()):
        if marker in line:
            return Path(line.split(marker, 1)[1].strip())
    return None

def run_single_model(model_name: str, stages: str = 'all'):
    cmd = [sys.executable, str(ROOT / 'run_model.py'), '--config', str(CONFIG), '--models', model_name, '--stages', stages]
    print('Running:', ' '.join(map(str, cmd)))
    proc = subprocess.Popen(cmd, cwd=ROOT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    lines = []
    for line in proc.stdout:
        print(line, end='')
        lines.append(line.rstrip(''))
    rc = proc.wait()
    if rc != 0:
        raise RuntimeError(f'Model run failed with code {rc}')
    run_dir = _parse_run_dir(''.join(lines))
    if run_dir is None:
        raise RuntimeError('Could not parse run directory from command output.')
    LAST_RUN_DIRS[model_name.upper()] = run_dir
    return run_dir

def get_run_dir(model_name: str):
    key = model_name.upper()
    if key in LAST_RUN_DIRS:
        return LAST_RUN_DIRS[key]
    candidates = sorted((ROOT / 'outputs').glob('run_*'))
    for run_dir in reversed(candidates):
        if (run_dir / 'r2' / f'{key.lower()}_r2_summary_full_large_small.csv').exists():
            LAST_RUN_DIRS[key] = run_dir
            return run_dir
    raise FileNotFoundError(f'No saved run found for {key}. Run the model first.')

def show_model_summary(model_name: str):
    key = model_name.lower()
    run_dir = get_run_dir(model_name)
    print('Run dir:', run_dir)
    display(pd.read_csv(ROOT / 'data' / 'panel_prep_summary.csv'))
    display(pd.read_csv(ROOT / 'data' / 'window_coverage_summary.csv'))
    display(pd.read_csv(run_dir / 'r2' / f'{key}_r2_summary_full_large_small.csv'))
    display(pd.read_csv(run_dir / 'portfolio' / f'{key}_performance_summary.csv'))
    display(pd.read_csv(run_dir / 'benchmark' / f'{key}_vs_benchmark.csv'))
    imp_path = run_dir / 'importance' / f'{key}_feature_importance.csv'
    if imp_path.exists():
        display(pd.read_csv(imp_path).head(15))
    comp_path = run_dir / 'complexity' / f'{key}_complexity.csv'
    if comp_path.exists():
        display(pd.read_csv(comp_path).head(15))

def show_latest_recommendations(model_name: str, top_k: int = 10, save_to_run_dir: bool = True):
    cfg = load_config(CONFIG)
    result = build_latest_recommendations(cfg, model_name, top_k=top_k)
    print('Latest fully labeled month scored:', result.latest_eom.date())
    print('Calibration window train:', result.train_start.date(), '->', result.train_end.date())
    print('Calibration window val  :', result.val_start.date(), '->', result.val_end.date())
    display(result.recommendations)
    if save_to_run_dir:
        run_dir = get_run_dir(model_name)
        out_path = run_dir / 'recommendations' / f"{model_name.lower()}_latest_recommendations.csv"
        out_path.parent.mkdir(parents=True, exist_ok=True)
        result.recommendations.to_csv(out_path, index=False)
        print('Saved:', out_path)

def show_true_latest_recommendations(model_name: str, top_k: int = 10, save_to_run_dir: bool = True):
    cfg = load_config(CONFIG)
    result = build_true_latest_recommendations(cfg, model_name, top_k=top_k)
    print('True latest month scored:', result.score_eom.date())
    print('Latest labeled month used for calibration:', result.latest_labeled_eom.date())
    print('Calibration window train:', result.train_start.date(), '->', result.train_end.date())
    print('Calibration window val  :', result.val_start.date(), '->', result.val_end.date())
    print('Eligible latest-month rows:', result.universe_rows)
    print('Scored latest-month rows  :', result.scored_rows)
    display(result.recommendations)
    if save_to_run_dir:
        run_dir = get_run_dir(model_name)
        out_path = run_dir / 'recommendations' / f"{model_name.lower()}_true_latest_recommendations.csv"
        out_path.parent.mkdir(parents=True, exist_ok=True)
        result.recommendations.to_csv(out_path, index=False)
        print('Saved:', out_path)


## OLS

Block 1 runs `OLS` and saves the full output bundle. Block 2 shows the quick evaluation tables. Block 3 scores the true latest month in `panel_input.csv` and shows the stocks this model would recommend for the next period.


In [ ]:
RUN_DIR_OLS = run_single_model('OLS')
RUN_DIR_OLS


In [ ]:
show_model_summary('OLS')


In [ ]:
show_true_latest_recommendations('OLS', top_k=10, save_to_run_dir=True)


## OLS3

Block 1 runs `OLS3` and saves the full output bundle. Block 2 shows the quick evaluation tables. Block 3 scores the true latest month in `panel_input.csv` and shows the stocks this model would recommend for the next period.


In [ ]:
RUN_DIR_OLS3 = run_single_model('OLS3')
RUN_DIR_OLS3


In [ ]:
show_model_summary('OLS3')


In [ ]:
show_true_latest_recommendations('OLS3', top_k=10, save_to_run_dir=True)


## ENET

Block 1 runs `ENET` and saves the full output bundle. Block 2 shows the quick evaluation tables. Block 3 scores the true latest month in `panel_input.csv` and shows the stocks this model would recommend for the next period.


In [ ]:
RUN_DIR_ENET = run_single_model('ENET')
RUN_DIR_ENET


In [ ]:
show_model_summary('ENET')


In [ ]:
show_true_latest_recommendations('ENET', top_k=10, save_to_run_dir=True)


## PLS

Block 1 runs `PLS` and saves the full output bundle. Block 2 shows the quick evaluation tables. Block 3 scores the true latest month in `panel_input.csv` and shows the stocks this model would recommend for the next period.


In [ ]:
RUN_DIR_PLS = run_single_model('PLS')
RUN_DIR_PLS


In [ ]:
show_model_summary('PLS')


In [ ]:
show_true_latest_recommendations('PLS', top_k=10, save_to_run_dir=True)


## PCR

Block 1 runs `PCR` and saves the full output bundle. Block 2 shows the quick evaluation tables. Block 3 scores the true latest month in `panel_input.csv` and shows the stocks this model would recommend for the next period.


In [ ]:
RUN_DIR_PCR = run_single_model('PCR')
RUN_DIR_PCR


In [ ]:
show_model_summary('PCR')


In [ ]:
show_true_latest_recommendations('PCR', top_k=10, save_to_run_dir=True)


## GBRT

Block 1 runs `GBRT` and saves the full output bundle. Block 2 shows the quick evaluation tables. Block 3 scores the true latest month in `panel_input.csv` and shows the stocks this model would recommend for the next period.


In [ ]:
RUN_DIR_GBRT = run_single_model('GBRT')
RUN_DIR_GBRT


In [ ]:
show_model_summary('GBRT')


In [ ]:
show_true_latest_recommendations('GBRT', top_k=10, save_to_run_dir=True)


## RF

Block 1 runs `RF` and saves the full output bundle. Block 2 shows the quick evaluation tables. Block 3 scores the true latest month in `panel_input.csv` and shows the stocks this model would recommend for the next period.


In [ ]:
RUN_DIR_RF = run_single_model('RF')
RUN_DIR_RF


In [ ]:
show_model_summary('RF')


In [ ]:
show_true_latest_recommendations('RF', top_k=10, save_to_run_dir=True)


## NN



Block 1 runs `NN` and saves the full output bundle. Block 2 shows the quick evaluation tables. Block 3 scores the true latest month in `panel_input.csv` and shows the stocks this model would recommend for the next period.


In [ ]:
RUN_DIR_NN = run_single_model('NN')

RUN_DIR_NN


In [ ]:
show_model_summary('NN')


In [ ]:
show_true_latest_recommendations('NN', top_k=10, save_to_run_dir=True)

